In [4]:
# Install dependencies (run this only once)
#pip install langchain langchain-openai openai psycopg2-binary pgvector psycopg2 tiktoken langchain_text_splitters 

from langchain_text_splitters  import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import psycopg2
from datetime import datetime

In [20]:
DB_CONN_INFO = {
    "host": "127.0.0.1",
    "dbname": "RagDB",
    "user": "postgres",
    "password": "1414",
    "port": 5432
}

In [15]:
# Split into coherent chunks
splitter = RecursiveCharacterTextSplitter(
        chunk_size=150,
        chunk_overlap=50,
        separators=["\n\n", "\n", "!", "?", ",", " ", ""]
    )
content = """
The Galaxy S26 is expected to feature a 6.3-inch Dynamic AMOLED 2x screen with 3000 nits peak brightness, an in-display ultrasonic fingerprint reader, a 50MP primary camera, a 50MP ultrawide camera, and a 10MP telephoto camera.
Samsung could equip the Galaxy S26 with the 2nm Exynos 2600 processor (Snapdragon 8 Elite Gen 5 in some countries), 12GB RAM, and 256GB/512GB storage. It will run Android 16-based One UI 8.5 out of the box with several new features and UI changes.
"""
chunks = splitter.split_text(content)
print(f"Chunks created: {len(chunks)}\n")
for i, c in enumerate(chunks, start=1):
    print(f"--- Chunk {i} ---\n{c}\n")

Chunks created: 4

--- Chunk 1 ---
The Galaxy S26 is expected to feature a 6.3-inch Dynamic AMOLED 2x screen with 3000 nits peak brightness, an in-display ultrasonic fingerprint reader

--- Chunk 2 ---
, an in-display ultrasonic fingerprint reader, a 50MP primary camera, a 50MP ultrawide camera, and a 10MP telephoto camera.

--- Chunk 3 ---
Samsung could equip the Galaxy S26 with the 2nm Exynos 2600 processor (Snapdragon 8 Elite Gen 5 in some countries), 12GB RAM

--- Chunk 4 ---
, 12GB RAM, and 256GB/512GB storage. It will run Android 16-based One UI 8.5 out of the box with several new features and UI changes.



In [ ]:
# Generate embeddings
embedder = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings = embedder.embed_documents(chunks)

In [22]:
# Connect to pgvector
conn = psycopg2.connect(**DB_CONN_INFO)
cur = conn.cursor()

In [23]:
# Ensure table exists
cur.execute("""
    CREATE TABLE IF NOT EXISTS documents (
        id SERIAL PRIMARY KEY,
        title TEXT,
        source TEXT,
        content TEXT,
        embedding vector(1536),
        timestamp TIMESTAMP
    )
""")

UndefinedObject: type "vector" does not exist
LINE 7:         embedding vector(1536),
                          ^


In [ ]:
# Insert chunks + metadata
title = "Samsung Galaxy S26"
source = "Samsung"
for chunk, vector in zip(chunks, embeddings):
    cur.execute("""
        INSERT INTO documents (title, source, content, embedding, timestamp)
        VALUES (%s, %s, %s, %s, %s)
    """, (title, source, chunk, vector, datetime.now()))

conn.commit()
cur.close()
conn.close()
print(f"✅ Indexed {len(chunks)} chunks from '{title}' into pgvector.")